# EXP-035 · Matched fold-3 pilot

Import this notebook into a **private** Kaggle notebook. Attach your private EXP-035 input dataset
and the pinned Qwen files as a separate private dataset or model. Select **GPU L4** if offered,
or **GPU T4 ×2**. The worker uses only GPU 0. Enable Internet to install the pinned Python environment.
Set `AUTHORIZE_RUN = True` below and click **Run All**. The preflight starts the existing EXP-035
worker, which continues to four epochs only if its measured forecast fits the five-hour ceiling.

The handoff requires no Kaggle or Hugging Face credentials inside the notebook.


In [ ]:
# Change the authorization before running on Kaggle.
AUTHORIZE_RUN = False
INPUT_ROOT = '/kaggle/input/exp035-fold3-inputs/exp035_inputs'  # Optional override for your actual mount.
MODEL_ROOT = '/kaggle/input/exp035-qwen'
RESUME_ARCHIVE = ""  # Optional: mounted exp035_results.zip or its extracted directory.
EXPECTED_MANIFEST = '599e255faa46565cd8225f05c9a81b1c1ba631bf4aba44bc99b89221e121e32a'
EXPECTED_ARCHIVE = '7f26ad070797f339c11de1f0cecaf55c0d91f31f6f69c5150852c66e6a276aed'


## 1. Find and verify the inputs

Extracted Kaggle datasets are discovered recursively. The original ZIP works too.
All bundled files are checked before the launcher is executed. The worker verifies the checkpoint
shards and processor against their original revision before loading Qwen.


In [ ]:
import hashlib
import json
import sys
import time
from pathlib import Path
from zipfile import ZipFile

if AUTHORIZE_RUN is not True:
    raise RuntimeError("Attach both private inputs, enable Kaggle GPU and Internet, then set AUTHORIZE_RUN = True and Run All.")
started = time.time()
kaggle_input = Path("/kaggle/input")
working = Path("/kaggle/working")
if not kaggle_input.is_dir() or not working.is_dir():
    raise RuntimeError("Run this notebook inside Kaggle.")
if sys.version_info < (3, 11):
    raise RuntimeError("This notebook launcher needs a Kaggle Python 3.11 or newer host.")

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def only(candidates, label):
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise ValueError(f"Expected one {label}, found {len(candidates)}. Attach the private input or set its exact mount in Settings.")
    return candidates[0]

handoff = Path(INPUT_ROOT)
if not (handoff / "exp035_handoff.json").is_file():
    unpacked = Path("/kaggle/temp/exp035_handoff/exp035_inputs")
    manifests = list(kaggle_input.rglob("exp035_handoff.json"))
    if manifests:
        handoff = only((p.parent for p in manifests), "EXP-035 handoff")
    else:
        archive = only(kaggle_input.rglob("exp035_fold3_inputs.zip"), "EXP-035 input ZIP")
        if sha256(archive) != EXPECTED_ARCHIVE:
            raise ValueError("Input ZIP SHA-256 mismatch. Attach the original EXP-035 archive.")
        with ZipFile(archive) as source:
            source.extractall(unpacked.parent)  # Only the exact, checksum-pinned ZIP can reach this line.
        handoff = unpacked

manifest_path = handoff / "exp035_handoff.json"
if sha256(manifest_path) != EXPECTED_MANIFEST:
    raise ValueError("Input manifest SHA-256 mismatch. Use the notebook and archive from the same handoff.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for name, expected in manifest["files"].items():
    path = handoff / name
    if not path.is_file() or sha256(path) != expected:
        raise ValueError(f"Missing or mismatched handoff file {name}. Reattach the verified private archive.")

checkpoint = Path(MODEL_ROOT)
if not all((checkpoint / name).is_file() for name in manifest["model"]["files"]):
    checkpoint = only((path.parent for path in kaggle_input.rglob("model.safetensors.index.json")
                       if all((path.parent / name).is_file() for name in manifest["model"]["files"])),
                      "complete pinned Qwen model folder including README.md")
print("Verified inputs and source", manifest["source"]["kaggle_revision"], flush=True)


## 2. Install, preflight and run the matched experiment

The launcher checks CUDA and FP16, verifies the sealed Ridge offsets, loads the pinned NF4 model,
checks parity and LoRA gradients, and measures throughput. It stops before training if the forecast
exceeds the remaining budget. Scientific epochs retain their original optimizer, loss and
inner-only checkpoint selection. Logs and sealed epochs appear under `/kaggle/working/exp035_fold3`.


In [ ]:
import subprocess
import sys

launcher = handoff / "runtime/scripts/exp035_kaggle.py"
command = [sys.executable, "-u", str(launcher), "--input-root", str(handoff),
           "--checkpoint", str(checkpoint), "--handoff-sha", EXPECTED_MANIFEST,
           "--started", str(started)]
if RESUME_ARCHIVE:
    command += ["--resume", RESUME_ARCHIVE]
subprocess.run(command, check=True)


## 3. Save results outside this session

Download **exp035_results.zip** and its **.sha256** from the Kaggle Output panel, or save a private
notebook version with outputs. The archive carries results or the exact stop reason and all
sealed epochs. To resume a controlled stop, attach the saved ZIP privately in a fresh Kaggle
session and set `RESUME_ARCHIVE` to the mounted ZIP or extracted directory. Use the same
accelerator model and handoff.
If Kaggle terminates a session abruptly, only artifacts exported or downloaded beforehand survive.


In [ ]:
from IPython.display import FileLink, display

for name in ("exp035_results.zip", "exp035_results.zip.sha256"):
    path = working / name
    if path.is_file():
        display(FileLink(str(path)))
status = working / "exp035_fold3/status.json"
if status.is_file():
    print(status.read_text(encoding="utf-8"))
preflights = sorted((working / "exp035_fold3").glob("preflight_*/result.json"))
if preflights:
    probe = json.loads(preflights[-1].read_text(encoding="utf-8"))
    sample_size = len(probe["sample"])
    print("Kaggle preflight", probe["state"], probe["hardware"]["gpu"],
          "warm B / live C paintings per second",
          round(sample_size / probe["frozen_cached_step_seconds"], 2),
          round(sample_size / probe["adapter_step_seconds"], 2),
          "allocated / reserved GiB",
          round(probe["peak_allocated_bytes"] / 2**30, 2),
          round(probe["peak_reserved_bytes"] / 2**30, 2))
metrics = working / "exp035_fold3/results/metrics.json"
if metrics.is_file():
    report = json.loads(metrics.read_text(encoding="utf-8"))
    for scope in ("full", "test_like"):
        for arm in ("prior", "frozen", "merger"):
            row = report[scope][arm]
            print(scope, arm, row["cultural_contrast"], row["log_loss"])
